# 08 — Controlled Imputation Experiments

## Objective

This notebook evaluates three candidate reconstruction methods for monthly environmental variables using **controlled artificial masking**:

1. **Linear interpolation**
2. **Seasonal reconstruction**
3. **K-nearest-neighbour (KNN) imputation**

The prepared monthly weather dataset contains no ordinary internal missing values. Therefore, known observations are temporarily hidden, reconstructed, and compared with their retained original values.

Two missingness patterns are evaluated:

- **Random isolated masking** at 5%, 10%, and 20% of eligible observations;
- **Contiguous temporal gaps** of 1, 2, and 3 months.

Reconstruction accuracy is evaluated using **MAE** and **RMSE**. Lower values indicate better reconstruction.

### Leakage protection

- January 2010 and December 2024 are excluded from the artificial-masking ground-truth pool because their weather coverage is incomplete.
- **2024 is excluded from imputation-method selection** because it is reserved for final forecasting evaluation.
- The benchmark is an **offline reconstruction experiment**. Linear and seasonal reconstruction may use observations occurring after a masked point within the pre-test period, so these results must not be interpreted as real-time forecasting performance.
- The original ground-truth dataframe is never overwritten.

## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import KNNImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

RANDOM_SEED = 42

In [2]:
PROJECT_ROOT = Path.cwd().parent
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
METRICS_DIR = RESULTS_DIR / "metrics"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Interim directory:", INTERIM_DIR)
print("Results directory:", RESULTS_DIR)

Project root: d:\Dengue_Project
Interim directory: d:\Dengue_Project\data\interim
Results directory: d:\Dengue_Project\results


## 2. Load and validate prepared monthly weather data

In [3]:
weather_path = INTERIM_DIR / "weather_monthly_prepared_2010_2024.csv"

print("Weather file:", weather_path)
print("File exists:", weather_path.exists())

if not weather_path.exists():
    raise FileNotFoundError(weather_path)

weather = pd.read_csv(weather_path, parse_dates=["date"])

print("Weather shape:", weather.shape)
weather.head()

Weather file: d:\Dengue_Project\data\interim\weather_monthly_prepared_2010_2024.csv
File exists: True
Weather shape: (4500, 16)


,date,year,month_number,administrative_district,temperature_mean_c,temperature_min_c,temperature_max_c,humidity_mean_percent,humidity_min_percent,humidity_max_percent,rainfall_total_mm,rainfall_max_hourly_mm,rainy_days,wind_speed_mean_kmh,wind_speed_max_kmh,complete_weather_month
0,2010-01-01,2010,1,Ampara,25.599833,22.20,28.95,80.270667,59.5,96.5,88.371429,4.35,21.714286,12.701000,24.15,False
1,2010-02-01,2010,2,Ampara,26.125893,22.20,31.40,78.548750,56.0,94.0,72.071429,4.35,12.571429,11.530714,24.15,True
2,2010-03-01,2010,3,Ampara,27.752097,22.55,32.55,75.846774,51.0,94.0,25.885714,2.45,14.500000,9.655000,21.65,True
3,2010-04-01,2010,4,Ampara,28.550833,25.20,32.80,79.705500,57.5,95.5,73.921429,3.45,28.214286,7.864833,16.35,True
4,2010-05-01,2010,5,Ampara,28.975968,25.45,34.20,79.241935,57.0,94.5,79.442857,4.95,28.000000,8.257419,28.35,True


In [4]:
experiment_variables = [
    "temperature_mean_c",
    "humidity_mean_percent",
    "rainfall_total_mm",
    "wind_speed_mean_kmh",
]

required_columns = [
    "administrative_district",
    "date",
    "complete_weather_month",
    *experiment_variables,
]

missing_columns = [c for c in required_columns if c not in weather.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Required columns verified.")
print("Standard missing values in experiment variables:")
print(weather[experiment_variables].isna().sum())

Required columns verified.
Standard missing values in experiment variables:
temperature_mean_c       0
humidity_mean_percent    0
rainfall_total_mm        0
wind_speed_mean_kmh      0
dtype: int64


In [5]:
assert weather.shape[0] == 4500, "Unexpected weather row count."
assert weather["administrative_district"].nunique() == 25
assert weather[experiment_variables].isna().sum().sum() == 0

print("Complete-weather flag counts:")
print(weather["complete_weather_month"].value_counts(dropna=False))

Complete-weather flag counts:
complete_weather_month
True     4450
False      50
Name: count, dtype: int64


## 3. Build protected ground truth and method-selection dataset

In [6]:
experiment_base = (
    weather.loc[weather["complete_weather_month"].eq(True),
                ["administrative_district", "date", *experiment_variables]]
    .copy()
    .sort_values(["administrative_district", "date"])
    .reset_index(drop=True)
)

print("Eligible complete-weather observations:", len(experiment_base))
print("Districts:", experiment_base["administrative_district"].nunique())
print("First eligible month:", experiment_base["date"].min())
print("Last eligible month:", experiment_base["date"].max())

months_per_district = experiment_base.groupby("administrative_district").size()
print("Minimum months per district:", months_per_district.min())
print("Maximum months per district:", months_per_district.max())

Eligible complete-weather observations: 4450
Districts: 25
First eligible month: 2010-02-01 00:00:00
Last eligible month: 2024-11-01 00:00:00
Minimum months per district: 178
Maximum months per district: 178


In [7]:
# Untouched source for artificial-mask ground truth.
ground_truth = experiment_base.copy()

# 2024 is protected from preprocessing-method selection.
method_selection_data = (
    ground_truth.loc[ground_truth["date"].dt.year <= 2023]
    .copy()
    .reset_index(drop=True)
)

print("Method-selection observations:", len(method_selection_data))
print("First month:", method_selection_data["date"].min())
print("Last month:", method_selection_data["date"].max())
print("Districts:", method_selection_data["administrative_district"].nunique())

assert len(ground_truth) == 4450
assert len(method_selection_data) == 4175
assert method_selection_data["date"].dt.year.max() == 2023
assert method_selection_data[experiment_variables].isna().sum().sum() == 0

print("2024 successfully excluded from method selection.")

Method-selection observations: 4175
First month: 2010-02-01 00:00:00
Last month: 2023-12-01 00:00:00
Districts: 25
2024 successfully excluded from method selection.


## 4. Experimental masking design

### Random isolated masking

Randomly selected interior observations are masked at **5%, 10%, and 20%**. The first and last observation of every district are protected. Adjacent observations are not selected together, so this scenario represents isolated missingness rather than accidental multi-month gaps.

### Contiguous temporal masking

For each district, three non-overlapping and non-touching gaps are generated for each gap length (**1, 2, and 3 months**). Every gap has an observed month before and after it, allowing linear interpolation to be evaluated as interpolation rather than boundary extrapolation.

The same mask is supplied to Linear, Seasonal, and KNN methods within each scenario.

In [8]:
random_masking_rates = [0.05, 0.10, 0.20]
temporal_gap_lengths = [1, 2, 3]
GAPS_PER_DISTRICT = 3

print("Random masking rates:", random_masking_rates)
print("Temporal gap lengths:", temporal_gap_lengths)
print("Gaps per district:", GAPS_PER_DISTRICT)

Random masking rates: [0.05, 0.1, 0.2]
Temporal gap lengths: [1, 2, 3]
Gaps per district: 3


In [9]:
def create_isolated_random_mask(data, variable, mask_rate, seed=42):
    """Create a reproducible random mask with no adjacent masked months."""
    local_rng = np.random.default_rng(seed)

    # Candidate rows are interior observations only.
    candidates = []
    position_lookup = {}

    for district in sorted(data["administrative_district"].unique()):
        district_data = (
            data.loc[data["administrative_district"].eq(district)]
            .sort_values("date")
        )
        district_indices = district_data.index.to_numpy()

        for pos in range(1, len(district_indices) - 1):
            idx = int(district_indices[pos])
            if pd.notna(data.loc[idx, variable]):
                candidates.append(idx)
                position_lookup[idx] = (district, pos)

    candidates = np.asarray(candidates, dtype=int)
    target_n = int(round(len(candidates) * mask_rate))
    shuffled = local_rng.permutation(candidates)

    selected = []
    selected_positions = {d: set() for d in data["administrative_district"].unique()}

    for idx in shuffled:
        district, pos = position_lookup[int(idx)]
        used = selected_positions[district]

        if (pos - 1 not in used) and (pos not in used) and (pos + 1 not in used):
            selected.append(int(idx))
            used.add(pos)

        if len(selected) == target_n:
            break

    if len(selected) != target_n:
        raise ValueError(
            f"Could create only {len(selected)} isolated masks; requested {target_n}."
        )

    return np.sort(np.asarray(selected, dtype=int))

In [10]:
def create_contiguous_mask(data, variable, gap_length, gaps_per_district=3, seed=42):
    """Create separated contiguous gaps with an observed month on both sides."""
    local_rng = np.random.default_rng(seed)
    selected_indices = []

    for district in sorted(data["administrative_district"].unique()):
        district_data = (
            data.loc[data["administrative_district"].eq(district)]
            .sort_values("date")
        )
        district_indices = district_data.index.to_numpy()
        n = len(district_indices)

        # Leave one observed position before and after every gap.
        possible_starts = list(range(1, n - gap_length))
        local_rng.shuffle(possible_starts)

        blocked_positions = set()
        gaps_created = 0

        for start in possible_starts:
            gap_positions = set(range(start, start + gap_length))
            protected_positions = set(range(start - 1, start + gap_length + 1))

            if gap_positions.isdisjoint(blocked_positions):
                gap_indices = district_indices[start:start + gap_length]

                # Only mask genuine observed values.
                if data.loc[gap_indices, variable].notna().all():
                    selected_indices.extend(gap_indices.tolist())
                    blocked_positions.update(protected_positions)
                    gaps_created += 1

            if gaps_created == gaps_per_district:
                break

        if gaps_created != gaps_per_district:
            raise ValueError(
                f"Could not create {gaps_per_district} gaps for {district}."
            )

    return np.sort(np.asarray(selected_indices, dtype=int))

In [11]:
print("Random isolated masks:")
for rate in random_masking_rates:
    idx = create_isolated_random_mask(
        method_selection_data,
        "temperature_mean_c",
        rate,
        RANDOM_SEED,
    )
    print(f"{rate * 100:.0f}% -> {len(idx)} observations")

print("\nContiguous masks:")
for gap_length in temporal_gap_lengths:
    idx = create_contiguous_mask(
        method_selection_data,
        "temperature_mean_c",
        gap_length,
        GAPS_PER_DISTRICT,
        RANDOM_SEED,
    )
    print(f"{gap_length}-month gaps -> {len(idx)} observations")

Random isolated masks:
5% -> 206 observations
10% -> 412 observations
20% -> 825 observations

Contiguous masks:
1-month gaps -> 75 observations
2-month gaps -> 150 observations
3-month gaps -> 225 observations


## 5. Reconstruction methods

### Linear interpolation
Performed independently within each administrative district.

### Seasonal reconstruction
A masked value is replaced by the mean of the **available observations from the same district and calendar month**. Because the target is already `NaN`, it does not contribute to its own seasonal estimate.

### Scaled KNN
KNN uses the four environmental variables jointly. Variables are standardized before distance calculation so that variables with larger numerical scales do not dominate the neighbour search. A fixed configuration of `n_neighbors=5` and distance weighting is used; it is not tuned against the final test year.

In [12]:
def linear_reconstruction(data, variable):
    reconstructed = data.copy()
    reconstructed[variable] = (
        reconstructed.groupby("administrative_district", group_keys=False)[variable]
        .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
    )
    return reconstructed


def seasonal_reconstruction(data, variable):
    reconstructed = data.copy()
    reconstructed["month_number"] = reconstructed["date"].dt.month

    seasonal_means = (
        reconstructed.groupby(["administrative_district", "month_number"])[variable]
        .transform("mean")
    )
    reconstructed[variable] = reconstructed[variable].fillna(seasonal_means)
    return reconstructed


def scaled_knn_reconstruction(data, variable, n_neighbors=5):
    reconstructed = data.copy()
    knn_variables = [
        "temperature_mean_c",
        "humidity_mean_percent",
        "rainfall_total_mm",
        "wind_speed_mean_kmh",
    ]

    knn_data = reconstructed[knn_variables].copy()

    scaler = StandardScaler()
    scaled_values = scaler.fit_transform(knn_data)
    scaled_df = pd.DataFrame(
        scaled_values,
        columns=knn_variables,
        index=reconstructed.index,
    )

    imputer = KNNImputer(n_neighbors=n_neighbors, weights="distance")
    imputed_scaled = imputer.fit_transform(scaled_df)
    imputed_original = scaler.inverse_transform(imputed_scaled)

    imputed_df = pd.DataFrame(
        imputed_original,
        columns=knn_variables,
        index=reconstructed.index,
    )
    reconstructed[variable] = imputed_df[variable]
    return reconstructed

## 6. Evaluation helpers

In [13]:
def calculate_reconstruction_metrics(true_data, reconstructed_data, mask_indices, variable):
    true_values = true_data.loc[mask_indices, variable]
    predicted_values = reconstructed_data.loc[mask_indices, variable]

    if predicted_values.isna().any():
        raise ValueError(f"{variable}: reconstructed values still contain NaN.")

    mae = mean_absolute_error(true_values, predicted_values)
    rmse = np.sqrt(mean_squared_error(true_values, predicted_values))
    return mae, rmse


def run_imputation_experiment(true_data, variable, mask_indices):
    """Apply one shared mask to all three methods and return MAE/RMSE."""
    masked_data = true_data.copy()
    masked_data.loc[mask_indices, variable] = np.nan

    reconstructed = {
        "Linear": linear_reconstruction(masked_data, variable),
        "Seasonal": seasonal_reconstruction(masked_data, variable),
        "KNN": scaled_knn_reconstruction(masked_data, variable, n_neighbors=5),
    }

    rows = []
    for method, reconstructed_data in reconstructed.items():
        mae, rmse = calculate_reconstruction_metrics(
            true_data,
            reconstructed_data,
            mask_indices,
            variable,
        )
        rows.append({"method": method, "MAE": mae, "RMSE": rmse})

    return pd.DataFrame(rows)

## 7. Sanity test before the full experiment

In [14]:
sanity_indices = create_isolated_random_mask(
    method_selection_data,
    "temperature_mean_c",
    0.10,
    RANDOM_SEED,
)

sanity_results = run_imputation_experiment(
    method_selection_data,
    "temperature_mean_c",
    sanity_indices,
)

print("Sanity-test masked observations:", len(sanity_indices))
sanity_results.sort_values("RMSE")

Sanity-test masked observations: 412


,method,MAE,RMSE
0,Linear,0.351502,0.465566
1,Seasonal,0.502955,0.646342
2,KNN,1.322331,1.972361


## 8. Full controlled experiment

The full benchmark contains:

- 4 environmental variables × 3 random masking rates × 3 methods = **36 random-scenario result rows**;
- 4 environmental variables × 3 temporal gap lengths × 3 methods = **36 temporal-gap result rows**;
- **72 result rows in total**.

A fixed seed is used so the experiment is reproducible. The same mask is used by all three methods within each scenario.

In [15]:
random_results_list = []

for variable in experiment_variables:
    for mask_rate in random_masking_rates:
        mask_indices = create_isolated_random_mask(
            method_selection_data,
            variable,
            mask_rate,
            RANDOM_SEED,
        )

        result = run_imputation_experiment(
            method_selection_data,
            variable,
            mask_indices,
        )
        result["variable"] = variable
        result["missingness_type"] = "Random isolated"
        result["mask_rate"] = mask_rate
        result["gap_length"] = np.nan
        result["masked_observations"] = len(mask_indices)
        random_results_list.append(result)

        print(f"Completed: {variable} | {mask_rate * 100:.0f}% random")

random_results = pd.concat(random_results_list, ignore_index=True)
print("Random experiment rows:", len(random_results))

Completed: temperature_mean_c | 5% random
Completed: temperature_mean_c | 10% random
Completed: temperature_mean_c | 20% random
Completed: humidity_mean_percent | 5% random
Completed: humidity_mean_percent | 10% random
Completed: humidity_mean_percent | 20% random
Completed: rainfall_total_mm | 5% random
Completed: rainfall_total_mm | 10% random
Completed: rainfall_total_mm | 20% random
Completed: wind_speed_mean_kmh | 5% random
Completed: wind_speed_mean_kmh | 10% random
Completed: wind_speed_mean_kmh | 20% random
Random experiment rows: 36


In [16]:
temporal_results_list = []

for variable in experiment_variables:
    for gap_length in temporal_gap_lengths:
        mask_indices = create_contiguous_mask(
            method_selection_data,
            variable,
            gap_length,
            GAPS_PER_DISTRICT,
            RANDOM_SEED,
        )

        result = run_imputation_experiment(
            method_selection_data,
            variable,
            mask_indices,
        )
        result["variable"] = variable
        result["missingness_type"] = "Temporal gap"
        result["mask_rate"] = np.nan
        result["gap_length"] = gap_length
        result["masked_observations"] = len(mask_indices)
        temporal_results_list.append(result)

        print(f"Completed: {variable} | {gap_length}-month gaps")

temporal_results = pd.concat(temporal_results_list, ignore_index=True)
print("Temporal experiment rows:", len(temporal_results))

Completed: temperature_mean_c | 1-month gaps
Completed: temperature_mean_c | 2-month gaps
Completed: temperature_mean_c | 3-month gaps
Completed: humidity_mean_percent | 1-month gaps
Completed: humidity_mean_percent | 2-month gaps
Completed: humidity_mean_percent | 3-month gaps
Completed: rainfall_total_mm | 1-month gaps
Completed: rainfall_total_mm | 2-month gaps
Completed: rainfall_total_mm | 3-month gaps
Completed: wind_speed_mean_kmh | 1-month gaps
Completed: wind_speed_mean_kmh | 2-month gaps
Completed: wind_speed_mean_kmh | 3-month gaps
Temporal experiment rows: 36


In [17]:
all_imputation_results = pd.concat(
    [random_results, temporal_results],
    ignore_index=True,
)[[
    "variable",
    "missingness_type",
    "mask_rate",
    "gap_length",
    "masked_observations",
    "method",
    "MAE",
    "RMSE",
]]

print("Total experiment rows:", len(all_imputation_results))
all_imputation_results.head(15)

Total experiment rows: 72


,variable,missingness_type,mask_rate,gap_length,masked_observations,method,MAE,RMSE
0,temperature_mean_c,Random isolated,0.05,NaN,206,Linear,0.330414,0.437908
1,temperature_mean_c,Random isolated,0.05,NaN,206,Seasonal,0.496738,0.634900
2,temperature_mean_c,Random isolated,0.05,NaN,206,KNN,1.366989,1.990793
3,temperature_mean_c,Random isolated,0.10,NaN,412,Linear,0.351502,0.465566
4,temperature_mean_c,Random isolated,0.10,NaN,412,Seasonal,0.502955,0.646342
5,temperature_mean_c,Random isolated,0.10,NaN,412,KNN,1.322331,1.972361
6,temperature_mean_c,Random isolated,0.20,NaN,825,Linear,0.350882,0.463005
7,temperature_mean_c,Random isolated,0.20,NaN,825,Seasonal,0.488360,0.633024
8,temperature_mean_c,Random isolated,0.20,NaN,825,KNN,1.284465,1.959164
9,humidity_mean_percent,Random isolated,0.05,NaN,206,Linear,2.387664,3.251635


In [18]:
assert len(random_results) == 36
assert len(temporal_results) == 36
assert len(all_imputation_results) == 72
assert all_imputation_results[["MAE", "RMSE"]].isna().sum().sum() == 0
assert all_imputation_results["MAE"].ge(0).all()
assert all_imputation_results["RMSE"].ge(0).all()
assert method_selection_data["date"].dt.year.max() == 2023

print("All full-experiment validation checks passed.")

All full-experiment validation checks passed.


## 9. Scenario-level winners and method comparison

In [19]:
scenario_winners = (
    all_imputation_results
    .sort_values("RMSE")
    .groupby(
        ["variable", "missingness_type", "mask_rate", "gap_length"],
        dropna=False,
        as_index=False,
    )
    .first()
)

scenario_winners[[
    "variable",
    "missingness_type",
    "mask_rate",
    "gap_length",
    "masked_observations",
    "method",
    "MAE",
    "RMSE",
]]

,variable,missingness_type,mask_rate,gap_length,masked_observations,method,MAE,RMSE
0,humidity_mean_percent,Random isolated,0.05,NaN,206,Linear,2.387664,3.251635
1,humidity_mean_percent,Random isolated,0.10,NaN,412,Linear,2.357511,3.195024
2,humidity_mean_percent,Random isolated,0.20,NaN,825,Linear,2.424221,3.243968
3,humidity_mean_percent,Temporal gap,NaN,1.0,75,KNN,2.721479,3.465946
4,humidity_mean_percent,Temporal gap,NaN,2.0,150,Seasonal,2.447482,3.097973
5,humidity_mean_percent,Temporal gap,NaN,3.0,225,Seasonal,2.672118,3.611751
6,rainfall_total_mm,Random isolated,0.05,NaN,206,KNN,57.504152,81.046888
7,rainfall_total_mm,Random isolated,0.10,NaN,412,KNN,60.907220,88.982686
8,rainfall_total_mm,Random isolated,0.20,NaN,825,KNN,58.027291,83.829280
9,rainfall_total_mm,Temporal gap,NaN,1.0,75,KNN,47.296719,68.331817


In [20]:
method_win_counts = (
    scenario_winners["method"]
    .value_counts()
    .rename_axis("method")
    .reset_index(name="number_of_wins")
)

method_win_counts

,method,number_of_wins
0,Seasonal,10
1,Linear,7
2,KNN,7


In [21]:
average_performance = (
    all_imputation_results
    .groupby(["variable", "method"], as_index=False)
    .agg(
        mean_MAE=("MAE", "mean"),
        mean_RMSE=("RMSE", "mean"),
    )
    .sort_values(["variable", "mean_RMSE"])
)

average_performance

,variable,method,mean_MAE,mean_RMSE
0,humidity_mean_percent,KNN,2.551230,3.446950
2,humidity_mean_percent,Seasonal,2.664603,3.497166
1,humidity_mean_percent,Linear,2.780306,3.744449
3,rainfall_total_mm,KNN,55.130979,79.192875
5,rainfall_total_mm,Seasonal,67.486840,97.866501
4,rainfall_total_mm,Linear,83.311007,112.993243
7,temperature_mean_c,Linear,0.409220,0.535286
8,temperature_mean_c,Seasonal,0.488764,0.628997
6,temperature_mean_c,KNN,1.368548,2.079613
11,wind_speed_mean_kmh,Seasonal,1.490169,2.000375


### Interpretation rule

Do **not** compare raw RMSE values across different environmental variables as though they share one unit. Temperature, humidity, rainfall, and wind speed have different physical scales.

The preferred method should therefore be interpreted **within each variable and scenario**. If different variables consistently favour different methods, a variable-specific reconstruction strategy is more defensible than forcing one global method.

In [22]:
best_average_method_by_variable = (
    average_performance
    .sort_values("mean_RMSE")
    .groupby("variable", as_index=False)
    .first()
)

best_average_method_by_variable

,variable,method,mean_MAE,mean_RMSE
0,humidity_mean_percent,KNN,2.551230,3.446950
1,rainfall_total_mm,KNN,55.130979,79.192875
2,temperature_mean_c,Linear,0.409220,0.535286
3,wind_speed_mean_kmh,Seasonal,1.490169,2.000375


## 10. Save reproducible research outputs

In [23]:
output_files = {
    "imputation_experiment_results.csv": all_imputation_results,
    "imputation_scenario_winners.csv": scenario_winners,
    "imputation_average_performance.csv": average_performance,
    "imputation_method_win_counts.csv": method_win_counts,
    "imputation_best_method_by_variable.csv": best_average_method_by_variable,
}

for filename, dataframe in output_files.items():
    output_path = TABLES_DIR / filename
    dataframe.to_csv(output_path, index=False)
    print(filename, "->", output_path.exists())

imputation_experiment_results.csv -> True
imputation_scenario_winners.csv -> True
imputation_average_performance.csv -> True
imputation_method_win_counts.csv -> True
imputation_best_method_by_variable.csv -> True


## 11. Methodological conclusion

After running the notebook, interpret the generated tables using the following rules:

- Lower **MAE** and **RMSE** indicate better reconstruction.
- Give primary attention to **RMSE within the same variable/scenario**, while using MAE as a complementary measure.
- Do not select a method merely because it has the largest total number of wins; variables have different units and error scales.
- If one method is consistently strongest for a particular environmental variable, that method may be selected for that variable.
- If performance differs by missingness pattern, report that sensitivity rather than hiding it.
- The population gap from 2010–2013 is **structural source unavailability** and is not addressed by these weather-imputation experiments.
- The 2024 period remains excluded from method selection and reserved for final forecasting evaluation.

The final preprocessing decision should be recorded only after reviewing `scenario_winners`, `average_performance`, and `best_average_method_by_variable`.

## Key Findings

1. The prepared monthly weather dataset contained no ordinary internal missing values. Therefore, controlled artificial masking was used to evaluate missing value reconstruction methods.

2. January 2010 and December 2024 were excluded from the artificial masking ground truth pool because these months had incomplete calendar day weather coverage.

3. The year 2024 was excluded from imputation method selection to preserve it as an untouched final forecasting test period.

4. The controlled experiment used eligible weather observations from February 2010 to December 2023 across 25 administrative districts.

5. Three reconstruction methods were evaluated: Linear interpolation, Seasonal reconstruction and scaled KNN imputation.

6. Two missingness patterns were considered: random isolated missingness at 5%, 10% and 20% and contiguous temporal gaps of 1, 2 and 3 months.

7. Identical artificial masks were used across competing methods to ensure a fair comparison.

8. Reconstruction accuracy was evaluated using MAE and RMSE, where lower values indicate better reconstruction performance.

9. Preliminary temperature experiments showed that Linear interpolation produced lower reconstruction error than Seasonal reconstruction and scaled KNN.

10. However, the final preprocessing strategy must be selected from the complete 72-result experiment because the best reconstruction method may differ among temperature, humidity, rainfall and wind speed.

11. Imputation method selection is based only on pre-2024 data, preventing the final 2024 forecasting test period from influencing preprocessing selection.

12. The controlled imputation experiment evaluates reconstruction ability and should not be interpreted as forecasting performance. Final forecasting performance will be evaluated separately using the fixed LSTM model.